# Precipitation by tracked feature class: monthly and seasonal maps

Maps of how much of the precipitation falls in MCS, multi-cell, isolated-cell and non-convective tracked features, made from the class maps of
`runscripts/run_trackpcp.py --steps 3` (or `Analysis/calc_monthly_rainmap_by_class.py`): hourly MRMS QPE matched to the 5-minute tracks of the radar-based MCS tracking.

**Definitions** (see `pyflextrkr/trackpcp_func.py`)
- A QPE record valid at time T covers the hour (T - 1 h, T]; the track masks of the 12 five-minute steps in that hour are merged into the track's *swath*.
- A pixel claimed by more than one track in an hour belongs to one track only: the best class (MCS first), then the track that covers it in most steps.
- **Frequency** = share of the hours in the month in which the pixel is in a feature of the class and the precipitation is at least the threshold of the data file (`pcp_thresh`, 1 mm/h).
  **Amount** = precipitation accumulated in those features (mm per month). **Share** = amount / total precipitation of the month (masked where the month's total is small).
- "Non-convective" is the class of tracked echo without convective cores (stratiform-only), as named in the class variable of the statistics file; class names and values are read from the files.

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature

DATA_DIR = "/global/cfs/cdirs/m1657/zfeng/MRMS/cell_mcs_conus/trackpcp/"   # trackpcp_classmaps_START_END.nc, one per month (run_trackpcp.py --steps 3)
FIG_DIR = DATA_DIR + "figures/"
os.makedirs(FIG_DIR, exist_ok=True)
SEASON_SUBSET = ["05", "06", "07", "08"]       # May-Aug ("May-Sug" of the request); the other season mean uses every month found
MIN_TOTAL_MM = 10.0                            # share of the total is not drawn where the total is below this [mm]

# HRRR Lambert Conformal as in conus_tracking/scripts/plot_subset_cell_mcs_tracks_terrain.py
PROJ = ccrs.LambertConformal(central_longitude=-97.5, central_latitude=38.5, standard_parallels=(38.5,))
DATA_CRS = ccrs.PlateCarree()
STATES = cfeature.NaturalEarthFeature("cultural", "admin_1_states_provinces_lakes", "50m")
INK, MUTED = "#222222", "#6b6b6b"
# categorical colours of the classes, in the fixed slot order of the reference palette (identity only: the maps use one-hue ramps)
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
OUTSIDE_COLOR = "#c9c8c2"                      # precipitation outside any tracked feature (de-emphasis grey)
NOCOV_COLOR = "#e9e8e3"                        # precipitation where the tracking input (composite reflectivity) has no data
mpl.rcParams.update({"font.size": 11, "axes.titlesize": 11, "text.color": INK, "axes.labelcolor": INK})

In [ ]:
# one file per month; the file name carries the tracking period
files = sorted(glob.glob(DATA_DIR + "trackpcp_classmaps_*.nc"))
assert files, f"no class map files in {DATA_DIR}"
months = {}
for f in files:
    m = re.search(r"classmaps_(\d{4})(\d{2})\d{2}\.", os.path.basename(f))
    months[m.group(1) + m.group(2)] = xr.open_dataset(f).load()
keys = sorted(months)
first = months[keys[0]]
class_values = [int(v) for v in first["class_value"].values]
class_names = [str(n) for n in first["class_name"].values]
print("months:", keys)
print("classes:", dict(zip(class_values, class_names)))
lon = first["longitude"].values
lon = np.where(lon > 180, lon - 360, lon)
lat = first["latitude"].values
xy = PROJ.transform_points(DATA_CRS, lon, lat)
EXTENT = [np.nanmin(xy[..., 0]), np.nanmax(xy[..., 0]), np.nanmin(xy[..., 1]), np.nanmax(xy[..., 1])]   # whole data domain
month_label = {k: pd.Timestamp(k + "01").strftime("%b %Y") for k in keys}

In [ ]:
# derived fields, [class, y, x] per month
def share_ok(total, uncovered):
    # the share is drawn where the total is not small and the pixel is mostly inside the coverage of the tracking input
    ok = total >= MIN_TOTAL_MM
    if uncovered is not None:
        ok = ok & (uncovered < 0.5 * np.where(total > 0, total, np.nan))
    return ok

def get_unc(ds):
    return ds["precip_uncovered"].values.astype(np.float64) if "precip_uncovered" in ds else None

def derive(ds, nwin=None):
    n = float(ds["ntimes"]) if nwin is None else nwin
    precip = ds["precip"].values.astype(np.float64)
    count = ds["precip_count"].values.astype(np.float64)
    total = ds["total_precip"].values.astype(np.float64)
    ok = share_ok(total, get_unc(ds))
    return {"freq": 100.0 * count / n, "amount": precip, "share": np.where(ok, 100.0 * precip / np.where(total > 0, total, np.nan), np.nan)}

monthly = {k: derive(months[k]) for k in keys}

def season(sub):
    # window-weighted mean over the months in sub: frequency from summed counts, amount = mean per month, share from summed precipitation
    nwin = sum(float(months[k]["ntimes"]) for k in sub)
    precip = sum(months[k]["precip"].values.astype(np.float64) for k in sub)
    count = sum(months[k]["precip_count"].values.astype(np.float64) for k in sub)
    total = sum(months[k]["total_precip"].values.astype(np.float64) for k in sub)
    uncs = [get_unc(months[k]) for k in sub]
    unc = sum(uncs) if all(u is not None for u in uncs) else None
    ok = share_ok(total / len(sub), None if unc is None else unc / len(sub))
    return {"freq": 100.0 * count / nwin, "amount": precip / len(sub),
            "share": np.where(ok, 100.0 * precip / np.where(total > 0, total, np.nan), np.nan)}

VARS = [  # key, panel title, colorbar label, colormap (one hue, light to dark)
    ("freq", "Precipitation frequency", "% of hours", "Purples"),
    ("amount", "Precipitation amount", "mm per month", "Blues"),
    ("share", "Share of total precipitation", "%", "Oranges"),
]

In [ ]:
def nice_max(x):
    # rounded-up maximum of the colour scale from a high percentile
    v = np.nanpercentile(x[np.isfinite(x) & (x > 0)], 99.5) if np.isfinite(x).any() else 1.0
    mag = 10 ** np.floor(np.log10(v))
    return float(np.ceil(v / mag * 2) / 2 * mag)

def draw_map(ax, field, vmax, cmap):
    ax.set_extent(EXTENT, crs=PROJ)
    ax.set_facecolor("white")
    z = np.ma.masked_where(~np.isfinite(field) | (field <= 0), field)
    pm = ax.pcolormesh(lon, lat, z, transform=DATA_CRS, cmap=cmap, vmin=0, vmax=vmax, shading="auto", rasterized=True, zorder=2)
    ax.add_feature(STATES, facecolor="none", edgecolor="#9a9a9a", linewidth=0.3, zorder=3)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), edgecolor="#777777", linewidth=0.4, zorder=3)
    ax.spines["geo"].set_edgecolor("#bbbbbb")
    ax.spines["geo"].set_linewidth(0.6)
    return pm

def monthly_figure(ci, fname=None):
    # rows = months, columns = frequency / amount / share of total; one colour bar per column (scale shared by the months)
    nrow, ncol = len(keys), len(VARS)
    fig = plt.figure(figsize=(14.2, 2.75 * nrow + 1.1), dpi=100)
    H = 2.75 * nrow + 1.1
    gs = gridspec.GridSpec(nrow + 1, ncol, figure=fig, height_ratios=[1] * nrow + [0.07], hspace=0.06, wspace=0.03,
                           top=1 - 0.75 / H, bottom=0.6 / H)
    pms = []
    for c, (vk, title, cblabel, cmapname) in enumerate(VARS):
        vmax = nice_max(np.concatenate([monthly[k][vk][ci].ravel() for k in keys]))
        cmap = mpl.colormaps[cmapname]
        for r, k in enumerate(keys):
            ax = fig.add_subplot(gs[r, c], projection=PROJ)
            pm = draw_map(ax, monthly[k][vk][ci], vmax, cmap)
            if r == 0:
                ax.set_title(title, fontsize=12, color=INK, pad=6)
            if c == 0:
                ax.text(-0.01, 0.5, month_label[k], transform=ax.transAxes, rotation=90, va="center", ha="right", fontsize=11, color=INK)
        cax = fig.add_subplot(gs[nrow, c])
        cb = fig.colorbar(pm, cax=cax, orientation="horizontal", extend="max")
        cb.set_label(cblabel, color=INK)
        cb.outline.set_edgecolor("#bbbbbb")
    fig.suptitle(f"{class_names[ci]} (class {class_values[ci]}): precipitation in tracked features, hourly QPE", y=1 - 0.2 / H, fontsize=13, fontweight="bold", color=INK)
    if fname:
        fig.savefig(FIG_DIR + fname, dpi=150, bbox_inches="tight", facecolor="white")
    return fig

def seasonal_figure(sub, label, fname=None):
    # rows = variables, columns = classes; one colour bar per row (scale shared by the classes)
    s = season(sub)
    nrow, ncol = len(VARS), len(class_values)
    fig = plt.figure(figsize=(4.3 * ncol + 1.0, 2.75 * nrow + 0.8), dpi=100)
    gs = gridspec.GridSpec(nrow, ncol + 1, figure=fig, width_ratios=[1] * ncol + [0.035], hspace=0.08, wspace=0.03, top=0.93)
    for r, (vk, title, cblabel, cmapname) in enumerate(VARS):
        vmax = nice_max(s[vk])
        cmap = mpl.colormaps[cmapname]
        for c in range(ncol):
            ax = fig.add_subplot(gs[r, c], projection=PROJ)
            pm = draw_map(ax, s[vk][c], vmax, cmap)
            if r == 0:
                ax.set_title(f"{class_names[c]}", fontsize=12, color=INK, pad=6)
            if c == 0:
                ax.text(-0.01, 0.5, title, transform=ax.transAxes, rotation=90, va="center", ha="right", fontsize=11, color=INK)
        cax = fig.add_subplot(gs[r, ncol])
        cb = fig.colorbar(pm, cax=cax, extend="max")
        cb.set_label(cblabel, color=INK)
        cb.outline.set_edgecolor("#bbbbbb")
    fig.suptitle(f"Precipitation by tracked feature class, {label}", y=0.985, fontsize=13, fontweight="bold", color=INK)
    if fname:
        fig.savefig(FIG_DIR + fname, dpi=150, bbox_inches="tight", facecolor="white")
    return fig

## Table view: domain totals

The same numbers as the maps, summed over the domain (mean depth over all pixels in mm, share of the total precipitation in %). The remainder to 100 % is precipitation outside
any tracked feature, and precipitation where the tracking input (composite reflectivity) has no data (offshore, Mexico: the QPE has values there, the reflectivity does not), which the tracking
cannot attribute; the last column gives the first class's share of the precipitation inside the coverage.

In [ ]:
rows = []
for k in keys + ["all months", "May-Aug"]:
    sub = keys if k == "all months" else ([x for x in keys if x[4:] in SEASON_SUBSET] if k == "May-Aug" else [k])
    if not sub:
        continue
    tot = sum(float(np.nansum(months[x]["total_precip"].values.astype(np.float64))) for x in sub)
    unc = sum(float(np.nansum(months[x]["precip_uncovered"].values.astype(np.float64))) for x in sub if "precip_uncovered" in months[x])
    npix = months[sub[0]]["total_precip"].size
    row = {"period": month_label.get(k, k), "domain-mean total (mm per month)": tot / npix / len(sub)}
    for ci, name in enumerate(class_names):
        row[f"{name} (%)"] = 100.0 * sum(float(months[x]["precip"].values[ci].astype(np.float64).sum()) for x in sub) / tot
    row["outside tracked features (%)"] = 100.0 * (tot - unc) / tot - sum(row[f"{n} (%)"] for n in class_names)
    row["outside reflectivity coverage (%)"] = 100.0 * unc / tot
    row[f"{class_names[0]} share within coverage (%)"] = row[f"{class_names[0]} (%)"] / (100.0 - row["outside reflectivity coverage (%)"]) * 100.0
    rows.append(row)
table = pd.DataFrame(rows).set_index("period").round(1)
table

In [ ]:
# part-to-whole: share of the total precipitation of the domain by class, one bar per period
share_cols = [f"{n} (%)" for n in class_names] + ["outside tracked features (%)", "outside reflectivity coverage (%)"]
fig, ax = plt.subplots(figsize=(10.5, 0.55 * len(table) + 1.6), dpi=100)
left = np.zeros(len(table))
ypos = np.arange(len(table))[::-1]
for j, col in enumerate(share_cols):
    vals = table[col].values
    color = (NOCOV_COLOR if "coverage" in col else OUTSIDE_COLOR) if col.startswith("outside") else CLASS_COLORS[j % len(CLASS_COLORS)]
    ax.barh(ypos, vals, left=left, height=0.55, color=color, edgecolor="white", linewidth=1.5, label=col.replace(" (%)", ""))
    for y, v, l in zip(ypos, vals, left):
        if v >= 6:                               # a label only where it fits
            ax.text(l + v / 2, y, f"{v:.0f}", ha="center", va="center", fontsize=10, color="white" if not col.startswith("outside") else INK)
    left += vals
ax.set_yticks(ypos)
ax.set_yticklabels(table.index, color=INK)
ax.set_xlim(0, 100)
ax.set_xlabel("Share of the total precipitation of the domain (%)")
for sp in ("top", "right", "left"):
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_color("#bbbbbb")
ax.tick_params(axis="y", length=0)
ax.xaxis.grid(True, color="#e6e6e6", linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False, fontsize=10)
fig.savefig(FIG_DIR + "class_share_by_month.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

## Monthly maps, one figure per class
Rows are months, columns are frequency, amount and share of the total precipitation. The colour scale of a column is shared by its months.

In [ ]:
figs = []
for ci in range(len(class_values)):
    fname = f"monthly_rainmaps_class{class_values[ci]}_{re.sub('[^A-Za-z0-9]+', '_', class_names[ci])}.png"
    figs.append(monthly_figure(ci, fname))
    plt.show()

## Seasonal means: every month found and May-Aug
Columns are the feature classes, rows the three variables (the colour scale of a row is shared by the classes).

In [ ]:
sub_all = keys
sub_mjja = [k for k in keys if k[4:] in SEASON_SUBSET]
f1 = seasonal_figure(sub_all, f"mean of {month_label[sub_all[0]]} - {month_label[sub_all[-1]]}", "seasonal_rainmaps_all_months.png")
plt.show()
if sub_mjja:
    f2 = seasonal_figure(sub_mjja, f"mean of {month_label[sub_mjja[0]]} - {month_label[sub_mjja[-1]]}", "seasonal_rainmaps_may_aug.png")
    plt.show()
else:
    print("no month of the subset", SEASON_SUBSET, "in the files")